# RAG ENSEA — Notebook final

Assistant documentaire local de l’ENSEA, enrichi par la bibliothèque économique ECO RCI. Il interroge automatiquement l’index documentaire principal, l’index des publications Facebook, ou les deux selon la question.

Ce notebook **ne relance ni le scraping ni l’indexation**. Il charge les index validés et fournit l’interface finale de question-réponse avec citations, cache et observabilité.

## Résultats de référence validés

- la référence historique avant ajout d’ECO RCI était de **4459 chunks documentaires** ;
- la référence Facebook était de **250 chunks**, issus de 227 publications nettoyées ;
- retrieval institutionnel : **MRR 0,7083**, **Hit@5 0,90** sur le test final élargi ;
- retrieval Facebook : **MRR 0,9375**, **Hit@3 et Hit@5 1,00** sur 16 questions ;
- préservation du routage institutionnel : **100/100** questions ;
- génération routée : **6/6 routes**, **6/6 citations** et **100 % de couverture des faits contrôlés**.

Les nombres exacts de chunks sont maintenant contrôlés dynamiquement après la reconstruction. La fenêtre Ollama reste limitée à **4096 tokens**, le meilleur palier pour la RTX 5060 Laptop 8 Go : le palier 5120 a provoqué un débordement VRAM et une forte baisse de débit.

## 1. Initialisation de l’environnement

Démarrez Ollama avant l’exécution. Utilisez un kernel Python contenant les dépendances du projet, puis exécutez les cellules dans l’ordre.

In [1]:
from pathlib import Path
import logging
import sys
import time

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s:%(name)s:%(message)s',
)
print('Projet :', PROJECT_ROOT)

Projet : c:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs\RAG ENSEA


In [2]:
import torch
from config import load_config

config = load_config(PROJECT_ROOT / 'config.yaml')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU uniquement'
gpu_memory_gb = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
    if torch.cuda.is_available() else 0.0
)
gpu_free_gb = (torch.cuda.mem_get_info()[0] / 1024**3) if torch.cuda.is_available() else 0.0
print('GPU :', gpu_name)
print(f'VRAM : {gpu_memory_gb:.1f} Go')
print(f'VRAM libre avant chargement : {gpu_free_gb:.1f} Go')
print('LLM :', config.generation.model_name)
print('Fenêtre Ollama :', config.generation.num_ctx)
print('Routage Facebook :', config.facebook_scraping.retrieval_enabled)
if torch.cuda.is_available() and gpu_free_gb < 6.0:
    print('ATTENTION : VRAM déjà occupée. Arrêtez les autres kernels Jupyter avant de mesurer les performances.')
assert config.generation.num_ctx == 4096, 'La fenêtre validée pour ce GPU doit être 4096.'

GPU : NVIDIA GeForce RTX 5060 Laptop GPU
VRAM : 8.0 Go
VRAM libre avant chargement : 6.9 Go
LLM : qwen3.5:9b
Fenêtre Ollama : 4096
Routage Facebook : True


## 2. Chargement du pipeline final

Les modèles d’embedding et de reranking sont mutualisés entre les trois index : ENSEA, économique et Facebook. Le premier chargement est naturellement plus long ; les requêtes suivantes utilisent les caches en mémoire.

In [3]:
from rag_pipeline import RAGPipeline

load_started = time.perf_counter()
pipeline = RAGPipeline(config)
load_seconds = time.perf_counter() - load_started

document_chunks = pipeline.store.dense.count()
economic_chunks = pipeline.economic_store.dense.count() if pipeline.economic_store else 0
facebook_chunks = pipeline.facebook_store.dense.count() if pipeline.facebook_store else 0
expected_document_chunks = sum(1 for line in (config.project.data_path / 'chunks' / 'chunks.jsonl').read_text(encoding='utf-8').splitlines() if line.strip())
expected_economic_chunks = sum(1 for line in config.local_documents.chunks_full_path.read_text(encoding='utf-8').splitlines() if line.strip())
expected_facebook_chunks = sum(1 for line in config.facebook_scraping.chunks_candidate_full_path.read_text(encoding='utf-8').splitlines() if line.strip())
print(f'Pipeline chargé en {load_seconds:.2f} s')
print('Chunks institutionnels ENSEA :', document_chunks)
print('Chunks économiques ECO RCI :', economic_chunks)
print('Chunks Facebook :', facebook_chunks)
assert document_chunks == expected_document_chunks
assert pipeline.economic_store is not None and economic_chunks == expected_economic_chunks
assert pipeline.facebook_store is not None and facebook_chunks == expected_facebook_chunks
print('État : prêt')

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /BAAI/bge-m3/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 1801a733-55a5-4763-afa5-940c3162ce92)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /BAAI/bge-m3/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 42f693ca-e584-44a6-bd58-9828a0209f1b)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

INFO:FlagEmbedding.finetune.embedder.encoder_only.m3.runner:loading existing colbert_linear and sparse_linear---------
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
INFO:rag_pipeline:Index économique séparé chargé : 3521 chunks.
INFO:rag_pipeline:Index Facebook séparé chargé : 249 chunks.


Pipeline chargé en 40.38 s
Chunks institutionnels ENSEA : 4715
Chunks économiques ECO RCI : 3521
Chunks Facebook : 249
État : prêt


## 3. Interface finale de question-réponse

La réponse est affichée progressivement. À la fin, le notebook présente la route réellement utilisée, les temps mesurés et les passages cités. Une publication Facebook reste explicitement identifiée comme une source datée.

In [4]:
import pandas as pd
from IPython.display import Markdown, display
from source_router import route_question

def ask_ensea(question: str, *, stream: bool = True):
    question = question.strip()
    if not question:
        raise ValueError('La question ne peut pas être vide.')

    preview = route_question(question)
    print(f'Route prévue : {preview.route} — {preview.reason}')
    started = time.perf_counter()

    if stream:
        print('\nRéponse :\n')
        for token in pipeline.ask_stream(question):
            print(token, end='', flush=True)
        print('\n')
        response = pipeline.last_response
    else:
        response = pipeline.ask(question)
        display(Markdown(response.answer))

    elapsed = time.perf_counter() - started
    decision = pipeline.retriever.last_decision
    retrieval_stats = pipeline.retriever.last_timings
    generation_stats = pipeline.generator.last_generation_stats
    status = pd.Series({
        'route': decision.route,
        'raison': decision.reason,
        'retrieval_secondes': retrieval_stats.get('total_seconds', 0.0),
        'génération_secondes': generation_stats.get('total_seconds', 0.0),
        'total_secondes': elapsed,
        'cache': generation_stats.get('cache_hit', False),
        'mode_réponse': pipeline.generator.last_prompt_stats.get('answer_mode'),
        'mots_réponse': len(response.answer.split()) if response else 0,
        'tokens_générés': generation_stats.get('generated_tokens', 0),
        'tokens_par_seconde': generation_stats.get('tokens_per_second', 0.0),
        'fenêtre_ollama': pipeline.generator.last_prompt_stats.get('ollama_num_ctx'),
    })
    display(status.to_frame('valeur'))

    if response and response.citations:
        citation_rows = [
            {
                'citation': f'[{citation.index}]',
                'titre': citation.title,
                'section': citation.section,
                'url': citation.url,
                'score': round(citation.rerank_score, 4),
                'preuve': citation.passage,
            }
            for citation in response.citations
        ]
        display(pd.DataFrame(citation_rows))
    else:
        print('Aucune citation : le système s’est abstenu ou aucun passage fiable n’a été trouvé.')
    return response

### Posez votre question

Modifiez seulement la variable `question`, puis réexécutez les deux cellules suivantes.

In [7]:
question = "Quel est le niveau de l'IHPC en 2025 ?"

In [8]:
response = ask_ensea(question, stream=True)

Route prévue : economic — source économique explicite : ihpc

Réponse :



INFO:vectorstore:Cache RAM des vecteurs chargé : 3521 vecteurs (13.8 Mo).
INFO:retriever:Retrieval 33.811s | query=0.040s dense=0.032s bm25=0.003s fusion=0.000s vectors+MMR=0.849s rerank=32.886s | 4 résultat(s)
INFO:generator:Budget du prompt : {'retrieved_chunks': 4, 'included_chunks': 2, 'context_tokens': 2207, 'prompt_tokens': 3007, 'input_token_limit': 3072, 'reserved_output_tokens': 1024, 'answer_mode': 'direct', 'conversation_turns': 0, 'ollama_num_ctx': 4096}
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


L'Indice Harmonisé des Prix à la Consommation (IHPC) en décembre 2025 a enregistré une hausse de 1,4% par rapport à avril 2025, soit un niveau de 105,0 [2]. En mars 2026, l'IHPC est resté quasi-constant par rapport à avril 2026, avec un niveau de 105,0 [2]. En décembre 2025, l'IHPC était à 105,0, ce qui correspond à une hausse de 1,4% par rapport à avril 2025 [2].

INFO:generator:Génération Ollama streaming : {'cache_hit': False, 'total_seconds': 100.48508759999822, 'ollama_prompt_tokens': 4067, 'generated_tokens': 148, 'tokens_per_second': 4.056615218370064, 'load_seconds': 0.1328959, 'prompt_eval_seconds': 63.72771, 'generation_eval_seconds': 36.483618, 'server_total_seconds': 100.4827439, 'unaccounted_seconds': 0.1408636999982278}


,valeur
route,economic
raison,source économique explicite : ihpc
retrieval_secondes,33.81077
génération_secondes,100.485088
total_secondes,134.321357
cache,False
mode_réponse,direct
mots_réponse,67
tokens_générés,148
tokens_par_seconde,4.056615


,citation,titre,section,url,score,preuve
0,[2],TABLEAU DE BORD SOCIO ECONOMIQUE (TBSE) MARS 2026,TABLEAU DE BORD SOCIO ECONOMIQUE (TBSE) MARS 2...,/api/source-files/ANSTAT/TABLEAU%20DE%20BORD%2...,0.9496,## Page 12 \nANStat - Direction des Statistiq...


## 4. Vérification du routeur sans génération

Cette cellule est rapide : elle permet de contrôler la source qui serait interrogée avant de lancer Ollama.

In [ ]:
route_examples = [
    "Qui est le Directeur Général de l'ENSEA ?",
    "Quel webinaire du Data Science Club porte sur Kaggle ?",
    "Quel appel à communications a été publié pour CISEA 2026 ?",
]
route_rows = []
for example in route_examples:
    decision = route_question(example)
    route_rows.append({
        'question': example,
        'route': decision.route,
        'confiance': decision.confidence,
        'raison': decision.reason,
    })
display(pd.DataFrame(route_rows))

## 5. Test rapide optionnel

Le test est désactivé par défaut pour ne pas lancer trois générations à chaque ouverture. Passez `RUN_SMOKE_TESTS` à `True` uniquement lorsque vous voulez vérifier les trois routes de bout en bout.

In [ ]:
RUN_SMOKE_TESTS = False
smoke_questions = [
    "Qui est le Directeur Général de l'ENSEA ?",
    "Qui est la sixième finaliste du concours ENSEA d'éloquence ?",
    "Quel appel à communications a été publié pour CISEA 2026 ?",
]
if RUN_SMOKE_TESTS:
    for smoke_question in smoke_questions:
        print('\n' + '=' * 100)
        ask_ensea(smoke_question, stream=False)
else:
    print('Tests optionnels désactivés.')

## 6. Observabilité

Chaque appel est enregistré localement sans stocker la réponse complète. Le tableau suivant résume les événements récents et permet d’identifier les requêtes lentes ou les erreurs.

In [ ]:
from observability import load_traces, summarize_traces

events = load_traces(config.observability.traces_full_path, limit=500)
summary = summarize_traces(events)
display(pd.Series(summary).to_frame('valeur'))

recent_rows = [
    {
        'date_utc': event.get('timestamp_utc'),
        'question': event.get('question'),
        'route': event.get('metadata', {}).get('route'),
        'statut': event.get('status'),
        'citations': event.get('citation_count'),
        'total_secondes': event.get('total_seconds'),
        'cache': event.get('generation_stats', {}).get('cache_hit'),
    }
    for event in events[-10:]
]
display(pd.DataFrame(recent_rows))

## Maintenance et réévaluation

Le notebook final reste volontairement séparé des opérations de maintenance :

- `collecte_facebook.ipynb` : nouvelle collecte Apify ;
- `preparation_facebook_candidate.ipynb` : nettoyage et chunking Facebook ;
- `index_benchmark_facebook.ipynb` : reconstruction et validation de l’index Facebook ;
- `benchmark_retrieval.ipynb` et `validation_finale_test.ipynb` : non-régression institutionnelle ;
- `observabilite.ipynb` : analyse détaillée des traces.

Après toute modification du corpus ou des paramètres, exécutez les notebooks spécialisés avant de réutiliser ce notebook final.